# Power grid load forecasting

This notebook runs the modular pipeline: **OPSD** hourly load (default Germany), **Open-Meteo** temperature and weather, calendar features (including German holidays), HDD/CDD-style proxies, then baselines, **Ridge**, **HGBR**, **LightGBM quantiles**, a **nonnegative weighted ensemble** optimized on validation MSE, and a **SARIMAX** fixed-origin benchmark over the validation period.

**Note:** The public OPSD `latest` CSV is a dated snapshot; default dates (2017–2018) stay inside typical coverage. See `data/README.md` and the lab `README.md` for newer data URLs.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve() / "labs" / "power_grid_load_forecasting"
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.pipeline import run_power_grid_load_pipeline
from src.inference import load_validation_predictions
from src.visualization import plot_forecast_intervals, plot_metrics_heatmap, plot_residuals_by_hour

artifacts = run_power_grid_load_pipeline(PROJECT_ROOT)
artifacts.model_artifacts.metrics.round(4)

In [ ]:
preds = load_validation_predictions(artifacts.predictions_path)
plot_forecast_intervals(preds, horizon_h=24)
plot_residuals_by_hour(preds, horizon_h=24)
plot_metrics_heatmap(artifacts.model_artifacts.metrics)

## Interpretation

- **naive_calendar**: for 24 h, current load as a proxy for the same clock hour on the next day; for 168 h, uses the `load_lag_168h` feature when present.
- **hour_dow_median_train**: median load by hour-of-day and weekday from the training split (Berlin local time).
- **sarimax_multi_fixed_origin** (`horizon_h=0`): multi-step forecast of **realized** load over validation from a model fit on the tail of training; not aligned with the direct `t → t+h` regression targets but useful as a classical benchmark.
- **ensemble_nonneg_weights**: convex combination of naive, hour–dow median, Ridge, HGBR, and LightGBM median, weights fit to minimize validation MSE.
- **LightGBM quantile** rows report pinball losses and **interval_coverage_80** for the 10–90% band.